#### Reading posthoc CSV stats
Start 5/5/25. To gather all posthoc csvs exported, unite into one file

#### Setup


In [1]:
%pip install -r requirements.txt
from pathlib import Path
import os
import notebook_setup
info = notebook_setup.setup()

# Environment & Imports Setup
import matplotlib as matplotlib
%matplotlib inline 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from datetime import datetime
import itertools
import ast
##customs 
from helper_functions import *

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


Note: you may need to restart the kernel to use updated packages.


<Figure size 640x480 with 0 Axes>

#### Read and gather all datasets 

In [2]:
#Set/create save and load folder paths 
here = info["repo_root"]
print(f" Here: {here}")
results_location = Path(here).parents[0] / "results"
data_location = Path(here).parents[0] / "data"
print(f"Saving results in {results_location}. Data location is {data_location}")
csv_folder_most_recent = results_location/ f"analysis_CSV_output/" #folders that analysis output goes to
os.chdir(results_location)
print(os.getcwd())


 Here: c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\code
Saving results in c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results. Data location is c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\data
c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results


In [3]:
folders = []
with os.scandir(os.getcwd()) as dir_iter:
    for item in dir_iter:
        # print(item)
        if item.is_dir():
            folders.append(item)
folders

[<DirEntry 'analysis_CSV_output'>,
 <DirEntry 'CCG_compare_12_Feb_2026 w old_01_21_2026_new_01_31_2026'>,
 <DirEntry 'CCG_compare_12_Feb_2026 w old_01_21_2026_new_02_12_2026'>,
 <DirEntry 'CCG_compare_16_Feb_2026 w old_01_21_2026_new_02_16_2026'>,
 <DirEntry 'CCG_compare_24_Feb_2026 w old_01_21_2026_new_02_16_2026'>,
 <DirEntry 'CCG_compare_25_Feb_2026 w old_01_21_2026_new_02_16_2026'>,
 <DirEntry 'CCG_compare_26_Jan_2026 w old_01_21_2026_new_01_24_2026'>,
 <DirEntry 'CCG_compare_26_Jan_2026 w old_05_31_2025_new_01_23_2026'>,
 <DirEntry 'CCG_compare_26_Jan_2026 w old_05_31_2025_new_01_24_2026'>,
 <DirEntry 'CCG_compare_29_Jan_2026 w old_01_21_2026_new_01_24_2026'>,
 <DirEntry 'CCG_compare_31_Jan_2026 w old_01_21_2026_new_01_31_2026'>,
 <DirEntry 'CCG_comparison_25_Jan_2026'>,
 <DirEntry 'CCG_comparison_26_Jan_2026'>,
 <DirEntry 'date_sorted_figures'>,
 <DirEntry 'dff_baseline_zscore_norm_ensemble_detection_01-Dec-2025_5000 shuffles'>,
 <DirEntry 'dff_baseline_zscore_norm_ensemble_detec

#### Sort folders in directory 

In [4]:
#sort folders in dir by last edited
sorted_folders = sorted(folders, reverse = True, key = lambda entry: entry.stat().st_mtime)
sorted_folders[:10]

[<DirEntry 'analysis_CSV_output'>,
 <DirEntry 'supp_fig_4'>,
 <DirEntry 'revision_fig_1'>,
 <DirEntry 'fig_4'>,
 <DirEntry 'fig_3'>,
 <DirEntry 'supp_fig_2'>,
 <DirEntry 'fig_7'>,
 <DirEntry 'fig_5'>,
 <DirEntry 'date_sorted_figures'>,
 <DirEntry 'fig_6'>]

In [5]:
sorted_csv_storage = [c for c in sorted_folders if 'analysis_CSV_output' in c.name]
print(sorted_csv_storage)
most_recent_csv_storage = sorted_csv_storage[0]
most_recent_csv_storage

[<DirEntry 'analysis_CSV_output'>]


<DirEntry 'analysis_CSV_output'>

#### get CSV folders in dir

In [6]:
## navigate
os.chdir(most_recent_csv_storage)
folder = os.getcwd()
print(folder)
csv_files = [f for f in os.listdir(folder) if f.lower().endswith(".csv")]
print(f" {len(csv_files)} csv files found in folder")

c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results\analysis_CSV_output
 780 csv files found in folder


In [7]:
csv_files

['1_# Perseverative Errors_behav_posthoc MWU_06_Apr_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_06_May_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_16_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_18_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_19_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_24_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_25_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_27_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_28_Feb_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_28_Mar_2026.csv',
 '1_# Perseverative Errors_behav_posthoc MWU_29_Mar_2026.csv',
 '1_behavior_perseverative_error_anova_06_Apr_2026.csv',
 '1_behavior_perseverative_error_anova_06_May_2026.csv',
 '1_behavior_perseverative_error_anova_16_Mar_2026.csv',
 '1_behavior_perseverative_error_anova_18_Mar_2026.csv',
 '1_behavior_perseverative_error_anova_19_Mar_2026.csv',
 '1_behavior_persevera

#### find latest run of data 

In [8]:
def nearest(items, pivot):
    return min(items, key=lambda x: abs(x - pivot)) 
    
def extract_date_str(csv_str, suffix = '.csv', split_delim = '_', get_post_split = -3):
    return csv_str.split(suffix)[0].split(split_delim)[get_post_split:]

In [9]:
date_list = [" ".join(extract_date_str(x)) for x in csv_files ] #drop .csv, then get last 3 entries, but only if has any digit in str
dates_clean = [x for x in date_list if sum(i.isdigit() for i in x) > 5]
dates_clean[-10:]

['01 Apr 2026',
 '19 Mar 2026',
 '23 Feb 2026',
 '25 Feb 2026',
 '26 May 2026',
 '28 Mar 2026',
 '11 May 2026',
 '11 May 2026',
 '11 May 2026',
 '26 May 2026']

In [10]:
now = datetime.today()
print(now)

2026-05-31 10:23:38.265199


#### for each unique fig number, find the csvs with the min timedelta datetime, and collect

In [11]:
def store_csv_name_by_fig_num(csv_files, max_fig_num = 8):
    ''' To- create dict where key = fig num and val = list of csv names with dates in title'''
    num_store = {f: list() for f in range(max_fig_num)}
    ## ## loop throuhg all CSVs
    for f in csv_files:
        fig_n = f.split("_")[0]
        if all((i.isdigit() for i in fig_n)):        # print(fig_n)
            num_store[int(fig_n)].append(f)
    return num_store

In [12]:
num_store = store_csv_name_by_fig_num(csv_files)
num_store

{0: [],
 1: ['1_# Perseverative Errors_behav_posthoc MWU_06_Apr_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_06_May_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_16_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_18_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_19_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_24_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_25_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_27_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_28_Feb_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_28_Mar_2026.csv',
  '1_# Perseverative Errors_behav_posthoc MWU_29_Mar_2026.csv',
  '1_behavior_perseverative_error_anova_06_Apr_2026.csv',
  '1_behavior_perseverative_error_anova_06_May_2026.csv',
  '1_behavior_perseverative_error_anova_16_Mar_2026.csv',
  '1_behavior_perseverative_error_anova_18_Mar_2026.csv',
  '1_behavior_perseverative_error_anova_19_Mar_2026.c

#### create csv store


In [13]:
results_location

WindowsPath('c:/Users/13car/Dropbox/local_github_repos_personal/dlx56_mPFC_1p_SohalLab/results')

In [14]:
csv_store_folder = results_location / "figure_tables"
make_folder(csv_store_folder)

The folder 'c:\Users\13car\Dropbox\local_github_repos_personal\dlx56_mPFC_1p_SohalLab\results\figure_tables' already exists.


#### test combination using figure 3 concat:

In [15]:
#
def get_last_fig_csv_names(num_store:dict, fig_num:int, skip_flag = ['anova', 'cells active per trial'],**kwargs):
    fig_storage = num_store[fig_num]
    ##processing logic 
    closest_time, last_datetime_str = get_latest_csv_datetime(fig_storage,skip_flag = skip_flag)##get list of dates in csvs for figure of interest, then find closest
    current_files = get_fig_csv_matching_datetime(fig_storage, closest_time,skip_flag = skip_flag)
    return current_files, closest_time
    
#get list of dates in csvs for figure of interest, then find closest
def get_latest_csv_datetime(fig_storage:list,skip_flag:list = ["skip"], **kwargs):
    ''' To iterate over list of .csv filenames (with datetime tags embedderd) and find the closest embedded tag to the current time. 
    Skip_flag: list of strings to iterate through and not include in csv list if present'''
    now = datetime.today()
    valid_figs = [x for x in fig_storage if all([skip.lower() not in x.lower() for skip in skip_flag])] #skip_flag: list of str, verify all not present

    date_list_datetime = [datetime.strptime(" ".join(extract_date_str(x)),'%d %b %Y') for x in valid_figs]
    closest_time = min(date_list_datetime , key = lambda x: now- x )
    last_datetime_str = closest_time.strftime('%d_%b_%Y')#convert to str to match save formatting
    print(f"closest datetime: {closest_time}. last_datetime_str = {last_datetime_str}")
    return closest_time, last_datetime_str

def get_fig_csv_matching_datetime(fig_storage, closest_time=datetime.today(), skip_flag:list = ["skip"],):
    ''' To loop over input list of .csv filenames, and pick entries matching the previously found closest datetime'''
    current_files = list()
    #optional- filter list and ignore ones with certain content
    valid_figs = [x for x in fig_storage if all([skip.lower() not in x.lower() for skip in skip_flag])] #skip_flag: list of str, verify all not present
    # print(valid_figs)
    for f in valid_figs:## given closest datetime, iterate and extract 
        f_date = datetime.strptime(" ".join(extract_date_str(f)),'%d %b %Y')# #get string date and convert to datetime to compare against last known date entry
        if f_date == closest_time:
            print(f'matching date file: {f}')
            current_files.append(f)
    
    return current_files


In [32]:
def concat_clean_csv_df(current_files, **kwargs):
    current_dfs = []
    for f in current_files:
        df = pd.read_csv(f)
        df['csv_date'] = " ".join(extract_date_str(os.path.basename(f)))  # e.g. '26 May 2026'
        df['csv_filename'] = os.path.basename(f)  # optional, for full traceability
        current_dfs.append(df)
    combined_fig_df = pd.concat(current_dfs)

    cols_to_drop = ['g_1_child_x', 'g_1_child_y', 'y_is_nonnan', 'y_is_2_elem', 'hue_is_x_axis',
                    'group_1_order_pos', 'group_2_order_pos', 'g_2_child_x', 'g_2_child_y',
                    'tick_text', 'tick_pos', 'hue_group_1_locs', 'hue_group_2_locs', 'Unnamed: 0',
                    'g1_num_loc', 'g2_num_loc', 'g1_cat_loc', 'g2_cat_loc', 'max_group_loc_val', 'plot_name']
    combined_fig_df.drop([c for c in cols_to_drop if c in combined_fig_df.columns], axis=1, inplace=True)
    return combined_fig_df


In [33]:
get_latest_csv_datetime(num_store[1],skip_flag = ['anova', 'trial'])

closest datetime: 2026-05-25 00:00:00. last_datetime_str = 25_May_2026


(datetime.datetime(2026, 5, 25, 0, 0), '25_May_2026')

In [34]:
fig_num = 3
current_files, closest_time = get_last_fig_csv_names(num_store, fig_num)
combined_fig_df= concat_clean_csv_df(current_files)

combined_fig_df.group_1_n = combined_fig_df.group_1_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
combined_fig_df.group_2_n = combined_fig_df.group_2_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)

#key value store for old: new col name
clean_col_name_dict = {'category_compared_within': "Group of posthoc comparison", 
                       'group_1': "Group 1",
                       'group_2': "Group 2", 
                       'group_1_n': "Group 1 N",
                       'group_2_n': "Group 2 N", 
                       'group_1_mean': "Group 1 Mean",
                       'group_1_sem': "Group 1 SEM",
                       'group_2_mean': "Group 2 Mean",
                       'group_2_sem':"Group 2 SEM",
                       'test_name': "Name of Statistical Test",
                       'stat_result': "Test Result",
                       'pvalue': "Test p-value",
                       'categorical_subgroup': "Alternate name- group compared within",
                       'numeric_var': "Variable Compared between Groups",
                       'hue_var': "Variable labeling post-hoc group",
                       'x_category_var': "Categorical variable of plot (x-axis)",
                       'date_tag': "Date of figure creation",
                       'fig_name': "filename of source table",
                       'fig_num': "Figure number"
                      }

combined_fig_df.rename(clean_col_name_dict, axis = 1,inplace = True)
combined_fig_df

closest datetime: 2026-05-26 00:00:00. last_datetime_str = 26_May_2026
matching date file: 3_null_CCG uni-class pred- Train IA test RS - Early_RS_Correct ensemble_predict_posthoc cohen d_26_May_2026.csv
matching date file: 3_null_CCGP- Train on IA trials, Test on RS trials_Early stage ens_ _ccgp_all_posthoc cohen d_26_May_2026.csv


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,Test p-value,Alternate name- group compared within,Variable Compared between Groups,Variable labeling post-hoc group,Categorical variable of plot (x-axis),Date of figure creation,filename of source table,Figure number,csv_date,csv_filename
0,Early RS Error,WT VEH,Het VEH,10000,10000,94.3869,2.4220,20.0621,3.8066,robust_cohen_d,...,0.000000e+00,Early RS Error,value,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,26 May 2026,3_null_CCG uni-class pred- Train IA test RS - ...
1,Early RS Error,Het VEH,Het CLNZ,10000,10000,20.0621,3.8066,46.2551,2.0052,robust_cohen_d,...,0.000000e+00,Early RS Error,value,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,26 May 2026,3_null_CCG uni-class pred- Train IA test RS - ...
2,Early RS Error,Het VEH,Het postCLNZ,10000,10000,20.0621,3.8066,62.2139,2.7931,robust_cohen_d,...,0.000000e+00,Early RS Error,value,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,26 May 2026,3_null_CCG uni-class pred- Train IA test RS - ...
3,Early RS Correct,WT VEH,Het VEH,10000,10000,54.4617,2.1427,53.7853,2.4082,robust_cohen_d,...,3.833236e-01,Early RS Correct,value,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,26 May 2026,3_null_CCG uni-class pred- Train IA test RS - ...
4,Early RS Correct,Het VEH,Het CLNZ,10000,10000,53.7853,2.4082,65.6328,2.4321,robust_cohen_d,...,4.906965e-07,Early RS Correct,value,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,26 May 2026,3_null_CCG uni-class pred- Train IA test RS - ...
5,Early RS Correct,Het VEH,Het postCLNZ,10000,10000,53.7853,2.4082,75.1442,2.2828,robust_cohen_d,...,0.000000e+00,Early RS Correct,value,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,26 May 2026,3_null_CCG uni-class pred- Train IA test RS - ...
0,Early_IA_Correct,WT VEH,Het VEH,10000,10000,0.4457,0.0072,0.4566,0.0240,robust_cohen_d,...,2.701414e-01,Early_IA_Correct,accuracy,geno_day,enriched_group,26_May_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,26 May 2026,"3_null_CCGP- Train on IA trials, Test on RS tr..."
1,Early_IA_Correct,Het VEH,Het CLNZ,10000,10000,0.4566,0.0240,0.5675,0.0179,robust_cohen_d,...,8.133608e-08,Early_IA_Correct,accuracy,geno_day,enriched_group,26_May_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,26 May 2026,"3_null_CCGP- Train on IA trials, Test on RS tr..."
2,Early_IA_Correct,Het VEH,Het postCLNZ,10000,10000,0.4566,0.0240,0.5456,0.0142,robust_cohen_d,...,3.177390e-06,Early_IA_Correct,accuracy,geno_day,enriched_group,26_May_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,26 May 2026,"3_null_CCGP- Train on IA trials, Test on RS tr..."
3,Early_IA_Error,WT VEH,Het VEH,10000,10000,0.4873,0.0137,0.4175,0.0099,robust_cohen_d,...,2.659076e-09,Early_IA_Error,accuracy,geno_day,enriched_group,26_May_2026,"null_CCGP- Train on IA trials, Test on RS tria...",3,26 May 2026,"3_null_CCGP- Train on IA trials, Test on RS tr..."


In [19]:
csv_name = f"fig_{fig_num}_results.csv"
combined_fig_df.to_csv( csv_store_folder/csv_name)

#### Concat posthoc statistics from figures 3-7:

In [35]:
## as figs 1 and 2 don't use traditional posthoc test df outputting, skip that for now 
fig_start = 2
fig_end = 8
figs_to_concat = np.arange(fig_start, fig_end)
print(f" Combining csvs with fig nums: {figs_to_concat}")
all_fig_tables = []
for fig_num in figs_to_concat:
    current_files, closest_time = get_last_fig_csv_names(num_store, fig_num)
    combined_fig_df= concat_clean_csv_df(current_files)
    combined_fig_df.group_1_n = combined_fig_df.group_1_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
    combined_fig_df.group_2_n = combined_fig_df.group_2_n.str.replace("(", "", regex = False).str.replace(",)", "", regex=False)
    #key value store for old: new col name
    clean_col_name_dict = {'category_compared_within': "Group of posthoc comparison", 
                           'group_1': "Group 1",
                           'group_2': "Group 2", 
                           'group_1_n': "Group 1 N",
                           'group_2_n': "Group 2 N", 
                           'group_1_mean': "Group 1 Mean",
                           'group_1_sem': "Group 1 SEM",
                           'group_2_mean': "Group 2 Mean",
                           'group_2_sem':"Group 2 SEM",
                           'test_name': "Name of Statistical Test",
                           'stat_result': "Test Result",
                           'pvalue': "Test p-value",
                           'categorical_subgroup': "Alternate name- group compared within",
                           'numeric_var': "Variable Compared between Groups",
                           'hue_var': "Variable labeling post-hoc group",
                           'x_category_var': "Categorical variable of plot (x-axis)",
                           'date_tag': "Date of figure creation",
                           'fig_name': "filename of source table",
                           'fig_num': "Figure number"
                          }
    combined_fig_df.rename(clean_col_name_dict, axis = 1,inplace = True)
    all_fig_tables.append(combined_fig_df)
    csv_name = f"fig_{fig_num}_concat_results.csv"
    combined_fig_df.to_csv( csv_store_folder/ csv_name)
full_fig_table = pd.concat(all_fig_tables)


 Combining csvs with fig nums: [2 3 4 5 6 7]
closest datetime: 2026-05-26 00:00:00. last_datetime_str = 26_May_2026
matching date file: 2_s_supp_CCG + null 1_class pred-Train_Correct_Test_Error- Early_RS_Error ensemble_supp_predict_posthoc cohen d_26_May_2026.csv
matching date file: 2_s_supp_CCG + null 1_class pred-Train_Correct_Test_Error-Early_IA_Correct ensemble_supp_predict_posthoc cohen d_26_May_2026.csv
matching date file: 2_s_supp_CCG+null single class pred- Train IA test RS - Early_RS_Correct ensemble_supp_predict_posthoc cohen d_26_May_2026.csv
closest datetime: 2026-05-26 00:00:00. last_datetime_str = 26_May_2026
matching date file: 3_null_CCG uni-class pred- Train IA test RS - Early_RS_Correct ensemble_predict_posthoc cohen d_26_May_2026.csv
matching date file: 3_null_CCGP- Train on IA trials, Test on RS trials_Early stage ens_ _ccgp_all_posthoc cohen d_26_May_2026.csv
closest datetime: 2026-05-26 00:00:00. last_datetime_str = 26_May_2026
matching date file: 4_null CCGP (ear

In [37]:
full_fig_table

,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,Alternate name- group compared within,Variable Compared between Groups,Variable labeling post-hoc group,Categorical variable of plot (x-axis),Date of figure creation,filename of source table,Figure number,csv_date,csv_filename,comparison
0,Early RS Error,WT VEH,WT CLNZ,10000,10000,43.4938,2.5059,20.9541,1.9260,robust_cohen_d,...,Early RS Error,value,geno_day,actual,26_May_2026,supp_CCG + null 1_class pred-Train_Correct_Tes...,2_s,26 May 2026,2_s_supp_CCG + null 1_class pred-Train_Correct...,NaN
1,Early RS Error,WT VEH,Het VEH,10000,10000,43.4938,2.5059,47.5030,2.2992,robust_cohen_d,...,Early RS Error,value,geno_day,actual,26_May_2026,supp_CCG + null 1_class pred-Train_Correct_Tes...,2_s,26 May 2026,2_s_supp_CCG + null 1_class pred-Train_Correct...,NaN
2,Early RS Error,Het VEH,Het CLNZ,10000,10000,47.5030,2.2992,69.0767,3.1235,robust_cohen_d,...,Early RS Error,value,geno_day,actual,26_May_2026,supp_CCG + null 1_class pred-Train_Correct_Tes...,2_s,26 May 2026,2_s_supp_CCG + null 1_class pred-Train_Correct...,NaN
3,Early RS Error,Het VEH,Het postCLNZ,10000,10000,47.5030,2.2992,26.7662,2.7451,robust_cohen_d,...,Early RS Error,value,geno_day,actual,26_May_2026,supp_CCG + null 1_class pred-Train_Correct_Tes...,2_s,26 May 2026,2_s_supp_CCG + null 1_class pred-Train_Correct...,NaN
4,Early IA Error,WT VEH,WT CLNZ,10000,10000,11.5559,1.4846,27.7649,1.6703,robust_cohen_d,...,Early IA Error,value,geno_day,actual,26_May_2026,supp_CCG + null 1_class pred-Train_Correct_Tes...,2_s,26 May 2026,2_s_supp_CCG + null 1_class pred-Train_Correct...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,Late_IA,Het VEH,Het CLNZ,1000,1000,1.1944,0.1677,0.9702,0.1143,robust_cohen_d,...,Late_IA,DB_index,geno_day,ensemble,26_May_2026,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,Late_IA_v_Early_RS_Correct
2,Late_IA,Het VEH,Het postCLNZ,1000,1000,1.1944,0.1677,1.0737,0.1479,robust_cohen_d,...,Late_IA,DB_index,geno_day,ensemble,26_May_2026,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,Late_IA_v_Early_RS_Correct
3,Early_RS_Correct,WT VEH,Het VEH,1000,1000,0.9893,0.1293,1.1455,0.1412,robust_cohen_d,...,Early_RS_Correct,DB_index,geno_day,ensemble,26_May_2026,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,Late_IA_v_Early_RS_Correct
4,Early_RS_Correct,Het VEH,Het CLNZ,1000,1000,1.1455,0.1412,1.0002,0.1254,robust_cohen_d,...,Early_RS_Correct,DB_index,geno_day,ensemble,26_May_2026,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,26 May 2026,7_7_H_Autoencoder Late_IA_v_Early_RS_Correct D...,Late_IA_v_Early_RS_Correct


In [22]:
## save NON STANDARD tables to separate csv for review, then drop from main table
non_standard = full_fig_table[full_fig_table['filename of source table'].isna()]#.dropna(axis=1, how='all')
non_standard.to_csv(csv_store_folder / "non_standard_figure_tables.csv", index=False)
non_standard


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,Test Result,Test p-value,Alternate name- group compared within,Variable Compared between Groups,Variable labeling post-hoc group,Categorical variable of plot (x-axis),Date of figure creation,filename of source table,Figure number,comparison


In [23]:
full_fig_table = full_fig_table.dropna(subset=['filename of source table']).dropna(axis=1, how='all')
full_fig_table


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,Test Result,Test p-value,Alternate name- group compared within,Variable Compared between Groups,Variable labeling post-hoc group,Categorical variable of plot (x-axis),Date of figure creation,filename of source table,Figure number,comparison
0,Early RS Error,WT VEH,WT CLNZ,10000,10000,43.4938,2.5059,20.9541,1.9260,robust_cohen_d,[1.008574e+01 0.000000e+00 2.000000e-04],0.000000e+00,Early RS Error,value,geno_day,actual,26_May_2026,supp_CCG + null 1_class pred-Train_Correct_Tes...,2_s,NaN
1,Early RS Error,WT VEH,Het VEH,10000,10000,43.4938,2.5059,47.5030,2.2992,robust_cohen_d,[1.6672e+00 4.7740e-02 2.0000e-04],4.773708e-02,Early RS Error,value,geno_day,actual,26_May_2026,supp_CCG + null 1_class pred-Train_Correct_Tes...,2_s,NaN
2,Early RS Error,Het VEH,Het CLNZ,10000,10000,47.5030,2.2992,69.0767,3.1235,robust_cohen_d,[7.86647e+00 0.00000e+00 2.00000e-04],1.776357e-15,Early RS Error,value,geno_day,actual,26_May_2026,supp_CCG + null 1_class pred-Train_Correct_Tes...,2_s,NaN
3,Early RS Error,Het VEH,Het postCLNZ,10000,10000,47.5030,2.2992,26.7662,2.7451,robust_cohen_d,[8.18992e+00 0.00000e+00 2.00000e-04],1.110223e-16,Early RS Error,value,geno_day,actual,26_May_2026,supp_CCG + null 1_class pred-Train_Correct_Tes...,2_s,NaN
4,Early IA Error,WT VEH,WT CLNZ,10000,10000,11.5559,1.4846,27.7649,1.6703,robust_cohen_d,[1.025742e+01 0.000000e+00 2.000000e-04],0.000000e+00,Early IA Error,value,geno_day,actual,26_May_2026,supp_CCG + null 1_class pred-Train_Correct_Tes...,2_s,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,Late_IA,Het VEH,Het CLNZ,1000,1000,1.1944,0.1677,0.9702,0.1143,robust_cohen_d,[1.562e+00 5.914e-02 2.000e-04],5.914425e-02,Late_IA,DB_index,geno_day,ensemble,26_May_2026,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,Late_IA_v_Early_RS_Correct
2,Late_IA,Het VEH,Het postCLNZ,1000,1000,1.1944,0.1677,1.0737,0.1479,robust_cohen_d,[7.6298e-01 2.2274e-01 2.0000e-04],2.227380e-01,Late_IA,DB_index,geno_day,ensemble,26_May_2026,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,Late_IA_v_Early_RS_Correct
3,Early_RS_Correct,WT VEH,Het VEH,1000,1000,0.9893,0.1293,1.1455,0.1412,robust_cohen_d,[1.15339e+00 1.24380e-01 2.00000e-04],1.243753e-01,Early_RS_Correct,DB_index,geno_day,ensemble,26_May_2026,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,Late_IA_v_Early_RS_Correct
4,Early_RS_Correct,Het VEH,Het CLNZ,1000,1000,1.1455,0.1412,1.0002,0.1254,robust_cohen_d,[1.08773e+00 1.38360e-01 2.00000e-04],1.383580e-01,Early_RS_Correct,DB_index,geno_day,ensemble,26_May_2026,7_H_Autoencoder Late_IA_v_Early_RS_Correct DB ...,7,Late_IA_v_Early_RS_Correct


In [24]:
unique_figs= sorted(full_fig_table['filename of source table'].unique())
# unique_figs

## create dict mapping key (filename) to value (corresponding figure panel)
map_panel_to_filename = {'Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': "6G" ,
 'Autoencoder Early_IA_Correct_v_Late_IA DB index by ensembles': "7D",
 'Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': "5H" ,
 'Autoencoder Late_IA_v_Early_RS_Correct DB index by ensembles': "7H" ,
 'CCG single class pred- Train Correct test Error- Early_IA_Correct ensemble': "4C" ,
 'CCG single class pred- Train Correct test Error- Early_RS_Error ensemble': "4F" ,
 'CCG single class pred- Train IA test RS - Early_RS_Correct ensemble': "3C" ,
 'Early_IA_Correct_v_Early_RS_Correct SVM accuracy by ensem': "6C" ,
 'Early_IA_Correct_v_Late_IA SVM accuracy by ensem': "7C" ,
 'Early_IA_Error_v_Early_RS_Error SVM accuracy by ensem': "5C" ,
 'Late_IA_v_Early_RS_Correct SVM accuracy by ensem': "7G" ,
 'time-dep decoding - Early RS Correct ens- Early_IA_Correct_v_Early_RS_Correct': "6D" ,
 'time-dep decoding - Early RS Error ens- Early_IA_Error_v_Early_RS_Error': "5D"
                        }
map_supp_panel_to_file = { 'Mean % of cells active in stage that are in stage ensemble':"S1I",
                           'supp_wtclnz_pointplot_Mean event rate by phase':"S1C",
                           'supp_Proportion frames active per trial': "S1B",
    'supp_Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': "S2I" ,
 'supp_Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': "S2G" ,
 'supp_CCG single class pred- Train Correct test Error- Early_IA_Correct ensemble': "S2D",
 'supp_CCG single class pred- Train Correct test Error- Early_RS_Error ensemble': "S2E",
 'supp_CCG single class pred- Train IA test RS - Early_RS_Correct ensemble': "S2C"
                         }

legacy_map = {
    'null_CCGP- Train on IA trials, Test on RS trials_Early stage ens_ ': '3B',
    'null CCGP (early ens) train_correct_test_error': '4B',
    'null_CCG uni-class pred- Train IA test RS - Early_RS_Correct ensemble': '3C',
    'null_ CCG uni-class pred- Train Correct test Error- Early_IA_Correct ensemble': '4C',
    'null_CCG uni-class pred- Train Correct test Error- Early_RS_Error ensemble': '4F',
    'Supplement- VEH v CLNZ for Het & WT- ensemble proportion overlap': 'S1F',  
}


full_filename_panel_map = {**map_panel_to_filename, **map_supp_panel_to_file, **legacy_map}
full_filename_panel_map

{'Autoencoder Early_IA_Correct_v_Early_RS_Correct DB index by ensembles': '6G',
 'Autoencoder Early_IA_Correct_v_Late_IA DB index by ensembles': '7D',
 'Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles': '5H',
 'Autoencoder Late_IA_v_Early_RS_Correct DB index by ensembles': '7H',
 'CCG single class pred- Train Correct test Error- Early_IA_Correct ensemble': '4C',
 'CCG single class pred- Train Correct test Error- Early_RS_Error ensemble': '4F',
 'CCG single class pred- Train IA test RS - Early_RS_Correct ensemble': '3C',
 'Early_IA_Correct_v_Early_RS_Correct SVM accuracy by ensem': '6C',
 'Early_IA_Correct_v_Late_IA SVM accuracy by ensem': '7C',
 'Early_IA_Error_v_Early_RS_Error SVM accuracy by ensem': '5C',
 'Late_IA_v_Early_RS_Correct SVM accuracy by ensem': '7G',
 'time-dep decoding - Early RS Correct ens- Early_IA_Correct_v_Early_RS_Correct': '6D',
 'time-dep decoding - Early RS Error ens- Early_IA_Error_v_Early_RS_Error': '5D',
 'Mean % of cells active in stage th

#### prepare and save concat figure DF

In [25]:
def clean_pvalue_string(x:float): 
    if x < 0.0001:
        if x == 0:
            cleaned =    r"<< 1 x 10^15"
        else:
            cleaned =    f'{x:.2e}'.replace("e", " x 10^")
    else:
        cleaned = f"{x:.4f}"
    return cleaned

In [26]:
import re
panel_re = re.compile(r'^(\d+)_?([A-Z])_')

def extract_panel_id(name):
    if pd.isna(name):
        return None
    m = panel_re.match(name)
    return f"{m.group(1)}{m.group(2)}" if m else None

# step 1: regex extraction for panel-prefixed filenames
full_fig_table["Figure Panel"] = full_fig_table['filename of source table'].apply(extract_panel_id)

# what's actually there
print(full_fig_table['filename of source table'].unique())
# how many match the map
matched = full_fig_table['filename of source table'].isin(full_filename_panel_map.keys())
print(f"Matching: {matched.sum()} of {len(full_fig_table)}")
# what's missing from the map
print(set(full_fig_table['filename of source table'].dropna().unique()) - set(full_filename_panel_map.keys()))

print(f"Regex populated Figure Panel for: {full_fig_table['Figure Panel'].notna().sum()} of {len(full_fig_table)}")
print(full_fig_table[['filename of source table', 'Figure Panel']].drop_duplicates().sort_values('Figure Panel'))


['supp_CCG + null 1_class pred-Train_Correct_Test_Error- Early_RS_Error ensemble'
 'supp_CCG + null 1_class pred-Train_Correct_Test_Error-Early_IA_Correct ensemble'
 'supp_CCG+null single class pred- Train IA test RS - Early_RS_Correct ensemble'
 'null_CCG uni-class pred- Train IA test RS - Early_RS_Correct ensemble'
 'null_CCGP- Train on IA trials, Test on RS trials_Early stage ens_ '
 'null CCGP (early ens) train_correct_test_error'
 'null_ CCG uni-class pred- Train Correct test Error- Early_IA_Correct ensemble'
 'null_CCG uni-class pred- Train Correct test Error- Early_RS_Error ensemble'
 'CCG single class pred- Train IA test RS - whole_pop ensemble'
 '5C_SVM classifier accuracy Early_IA_Error_v_Early_RS_Error by ensemble'
 '5D_Early RS Error ensemble time-based classification of Early IA Error v Early RS Error'
 '5_FGH_Combined_Latent_space_and_DB_index_Early_IA_Error_v_Early_RS_Error'
 '5_H_Autoencoder Early_IA_Error_v_Early_RS_Error DB index by ensembles'
 '6C_SVM classifier accu

In [27]:

valid_panels = list(full_filename_panel_map.values())
print(f" Keeping rows with panel IDs: {valid_panels}")
full_fig_table["Figure Panel"]= full_fig_table['filename of source table'].replace(full_filename_panel_map)
full_fig_table = full_fig_table[full_fig_table["Figure Panel"].isin(valid_panels)]## make sure you only keep table with real panels of interest
full_fig_table.sample()


 Keeping rows with panel IDs: ['6G', '7D', '5H', '7H', '4C', '4F', '3C', '6C', '7C', '5C', '7G', '6D', '5D', 'S1I', 'S1C', 'S1B', 'S2I', 'S2G', 'S2D', 'S2E', 'S2C', '3B', '4B', '3C', '4C', '4F', 'S1F']


,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,Test p-value,Alternate name- group compared within,Variable Compared between Groups,Variable labeling post-hoc group,Categorical variable of plot (x-axis),Date of figure creation,filename of source table,Figure number,comparison,Figure Panel
3,Early RS Correct,WT VEH,Het VEH,10000,10000,54.4617,2.1427,53.7853,2.4082,robust_cohen_d,...,0.383324,Early RS Correct,value,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,NaN,3C


In [28]:

## TEST RELATED PREPROCESS
#Rename varaible names
variable_map = {'prop_active_frames': "% of frames with events", 
                 'mean_rate': "Normalized event rate",
                 'active_in_trial': '% of cells active per trial',
                 'value': '% of samples classified',
                'accuracy':'SVM accuracy',
                'mean_acc': 'Time-dependent Decoding Accuracy',
                'DB_index':'Davies-Bouldin Index of Latent Space activity',
               }
full_fig_table['Variable Compared between Groups'] = full_fig_table['Variable Compared between Groups'].map(variable_map)
#find cohen's d 
cohen_d_mask =  full_fig_table['Name of Statistical Test'] == 'robust_cohen_d'
full_fig_table.loc[cohen_d_mask, 'Test Result'] = full_fig_table.loc[cohen_d_mask, 'Test Result'].str.replace(" ", ",").apply(ast.literal_eval) #aplpy to cohen's d testsing
#clean/redo testing 
test_name_clean = {'robust_cohen_d': "Robust Cohen's d",
                   "permutation_test": "Permutation Test",
                   'MWU': "Mann-Whitney U Test",
                   'chi_squared':"Chi-Squared Test"}
full_fig_table['Name of Statistical Test'] = full_fig_table['Name of Statistical Test'].map(test_name_clean) #replace var name with rea lnames 
#for MWU/permutation tests, replace first space occurance with comma then literal eval
test_stat_spaceless_mask = (full_fig_table['Name of Statistical Test'] == 'Permutation Test') | (full_fig_table['Name of Statistical Test'] == 'Mann-Whitney U Test')
full_fig_table.loc[test_stat_spaceless_mask, 'Test Result']= full_fig_table.loc[test_stat_spaceless_mask, 'Test Result'].str.replace(" ", ",", n = 1).apply(ast.literal_eval)
#clean p-values

full_fig_table['Test Statistic Variable'] = full_fig_table['Name of Statistical Test'].map({"Robust Cohen's d": "Robust Cohen's d",
                                                                                            "Permutation Test": "Mean permutation group diff.",
                                                                                            "Mann-Whitney U Test": "U-statistic",
                                                                                            "Chi-Squared Test": "Chi-Squared"})
full_fig_table['Test Statistic Value'] = full_fig_table['Test Result'].apply(lambda x: x[0])
full_fig_table['Test p-value'] = full_fig_table['Test p-value'].apply(lambda x: clean_pvalue_string(x))
#bugfix- force timebin in comparison string to avoid excel results as dates 
time_dep_rows = full_fig_table['filename of source table'].str.contains("time-dep")
full_fig_table.loc[time_dep_rows,'Group of posthoc comparison']= "timebins: "+ full_fig_table.loc[time_dep_rows,'Group of posthoc comparison']
full_fig_table.loc[time_dep_rows,'Alternate name- group compared within']= "timebins: "+ full_fig_table.loc[time_dep_rows,'Alternate name- group compared within']
full_fig_table.head()


C:\Users\13car\AppData\Local\Temp\ipykernel_48184\2365410214.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  full_fig_table['Variable Compared between Groups'] = full_fig_table['Variable Compared between Groups'].map(variable_map)
C:\Users\13car\AppData\Local\Temp\ipykernel_48184\2365410214.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  full_fig_table['Name of Statistical Test'] = full_fig_table['Name of Statistical Test'].map(test_name_clean) #replace var name with rea lnames
C:\Users\13car\AppD

,Group of posthoc comparison,Group 1,Group 2,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Name of Statistical Test,...,Variable Compared between Groups,Variable labeling post-hoc group,Categorical variable of plot (x-axis),Date of figure creation,filename of source table,Figure number,comparison,Figure Panel,Test Statistic Variable,Test Statistic Value
0,Early RS Error,WT VEH,Het VEH,10000,10000,94.3869,2.4220,20.0621,3.8066,Robust Cohen's d,...,% of samples classified,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,NaN,3C,Robust Cohen's d,23.29704
1,Early RS Error,Het VEH,Het CLNZ,10000,10000,20.0621,3.8066,46.2551,2.0052,Robust Cohen's d,...,% of samples classified,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,NaN,3C,Robust Cohen's d,8.60975
2,Early RS Error,Het VEH,Het postCLNZ,10000,10000,20.0621,3.8066,62.2139,2.7931,Robust Cohen's d,...,% of samples classified,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,NaN,3C,Robust Cohen's d,12.62587
3,Early RS Correct,WT VEH,Het VEH,10000,10000,54.4617,2.1427,53.7853,2.4082,Robust Cohen's d,...,% of samples classified,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,NaN,3C,Robust Cohen's d,0.29676
4,Early RS Correct,Het VEH,Het CLNZ,10000,10000,53.7853,2.4082,65.6328,2.4321,Robust Cohen's d,...,% of samples classified,geno_day,actual,26_May_2026,null_CCG uni-class pred- Train IA test RS - Ea...,3,NaN,3C,Robust Cohen's d,4.89533


In [29]:
## update with supplementary table 2 final format 
#reorder then drop figs
new_col_order = ['Figure Panel', 'Group of posthoc comparison', 'Group 1', 'Group 2','Variable Compared between Groups',
                 'Group 1 N', 'Group 2 N', 'Group 1 Mean', 'Group 1 SEM', 'Group 2 Mean', 'Group 2 SEM',
                 'Name of Statistical Test', 'Test p-value','Test Statistic Variable','Test Statistic Value',
                 'Variable labeling post-hoc group',
                 'Categorical variable of plot (x-axis)', 
                 'filename of source table',
                 ]

## final supplement table map
col_rename_map = {'Variable Compared between Groups': 'Dependent Variable',
                  'Name of Statistical Test': 'Statistical Test',
                  'Test p-value': 'p-value'}
full_fig_table = full_fig_table.loc[:, new_col_order].rename(columns = col_rename_map)
full_fig_table['shorthand test name'] = full_fig_table['Statistical Test'].map({"Robust Cohen's d": "d",
                                                                                         "Mann-Whitney U Test": "U"})
                                                                                        
full_fig_table['Test stat., Name, Value']= full_fig_table.apply(lambda x: f'{x['shorthand test name']}={round(x["Test Statistic Value"],3)}', axis=1)
# full_fig_table['Grouping Variable ']= full_fig_table['Variable Compared between Groups']

full_fig_table

,Figure Panel,Group of posthoc comparison,Group 1,Group 2,Dependent Variable,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Statistical Test,p-value,Test Statistic Variable,Test Statistic Value,Variable labeling post-hoc group,Categorical variable of plot (x-axis),filename of source table,shorthand test name,"Test stat., Name, Value"
0,3C,Early RS Error,WT VEH,Het VEH,% of samples classified,10000,10000,94.3869,2.4220,20.0621,3.8066,Robust Cohen's d,<< 1 x 10^15,Robust Cohen's d,23.29704,geno_day,actual,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=23.297
1,3C,Early RS Error,Het VEH,Het CLNZ,% of samples classified,10000,10000,20.0621,3.8066,46.2551,2.0052,Robust Cohen's d,<< 1 x 10^15,Robust Cohen's d,8.60975,geno_day,actual,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=8.61
2,3C,Early RS Error,Het VEH,Het postCLNZ,% of samples classified,10000,10000,20.0621,3.8066,62.2139,2.7931,Robust Cohen's d,<< 1 x 10^15,Robust Cohen's d,12.62587,geno_day,actual,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=12.626
3,3C,Early RS Correct,WT VEH,Het VEH,% of samples classified,10000,10000,54.4617,2.1427,53.7853,2.4082,Robust Cohen's d,0.3833,Robust Cohen's d,0.29676,geno_day,actual,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=0.297
4,3C,Early RS Correct,Het VEH,Het CLNZ,% of samples classified,10000,10000,53.7853,2.4082,65.6328,2.4321,Robust Cohen's d,4.91 x 10^-07,Robust Cohen's d,4.89533,geno_day,actual,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=4.895
5,3C,Early RS Correct,Het VEH,Het postCLNZ,% of samples classified,10000,10000,53.7853,2.4082,75.1442,2.2828,Robust Cohen's d,<< 1 x 10^15,Robust Cohen's d,9.10322,geno_day,actual,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=9.103
0,3B,Early_IA_Correct,WT VEH,Het VEH,SVM accuracy,10000,10000,0.4457,0.0072,0.4566,0.0240,Robust Cohen's d,0.2701,Robust Cohen's d,0.61239,geno_day,enriched_group,"null_CCGP- Train on IA trials, Test on RS tria...",d,d=0.612
1,3B,Early_IA_Correct,Het VEH,Het CLNZ,SVM accuracy,10000,10000,0.4566,0.0240,0.5675,0.0179,Robust Cohen's d,8.13 x 10^-08,Robust Cohen's d,5.23761,geno_day,enriched_group,"null_CCGP- Train on IA trials, Test on RS tria...",d,d=5.238
2,3B,Early_IA_Correct,Het VEH,Het postCLNZ,SVM accuracy,10000,10000,0.4566,0.0240,0.5456,0.0142,Robust Cohen's d,3.18 x 10^-06,Robust Cohen's d,4.51423,geno_day,enriched_group,"null_CCGP- Train on IA trials, Test on RS tria...",d,d=4.514
3,3B,Early_IA_Error,WT VEH,Het VEH,SVM accuracy,10000,10000,0.4873,0.0137,0.4175,0.0099,Robust Cohen's d,2.66 x 10^-09,Robust Cohen's d,5.83690,geno_day,enriched_group,"null_CCGP- Train on IA trials, Test on RS tria...",d,d=5.837


In [30]:

#save fig table


cols_drop_in_save = ['Test Result', 
                     'Alternate name- group compared within', 
                     'comparison',
                     'Figure number',
                     'Date of figure creation',
                     'Variable labeling post-hoc group',
                     'Categorical variable of plot (x-axis)']
full_fig_table = full_fig_table.drop([c for c in cols_drop_in_save if c in full_fig_table.columns],axis = 1)
csv_name = f"all_figure_concat_results.csv"
full_fig_table.set_index("Figure Panel").to_csv( csv_store_folder/csv_name)
full_fig_table

,Figure Panel,Group of posthoc comparison,Group 1,Group 2,Dependent Variable,Group 1 N,Group 2 N,Group 1 Mean,Group 1 SEM,Group 2 Mean,Group 2 SEM,Statistical Test,p-value,Test Statistic Variable,Test Statistic Value,filename of source table,shorthand test name,"Test stat., Name, Value"
0,3C,Early RS Error,WT VEH,Het VEH,% of samples classified,10000,10000,94.3869,2.4220,20.0621,3.8066,Robust Cohen's d,<< 1 x 10^15,Robust Cohen's d,23.29704,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=23.297
1,3C,Early RS Error,Het VEH,Het CLNZ,% of samples classified,10000,10000,20.0621,3.8066,46.2551,2.0052,Robust Cohen's d,<< 1 x 10^15,Robust Cohen's d,8.60975,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=8.61
2,3C,Early RS Error,Het VEH,Het postCLNZ,% of samples classified,10000,10000,20.0621,3.8066,62.2139,2.7931,Robust Cohen's d,<< 1 x 10^15,Robust Cohen's d,12.62587,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=12.626
3,3C,Early RS Correct,WT VEH,Het VEH,% of samples classified,10000,10000,54.4617,2.1427,53.7853,2.4082,Robust Cohen's d,0.3833,Robust Cohen's d,0.29676,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=0.297
4,3C,Early RS Correct,Het VEH,Het CLNZ,% of samples classified,10000,10000,53.7853,2.4082,65.6328,2.4321,Robust Cohen's d,4.91 x 10^-07,Robust Cohen's d,4.89533,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=4.895
5,3C,Early RS Correct,Het VEH,Het postCLNZ,% of samples classified,10000,10000,53.7853,2.4082,75.1442,2.2828,Robust Cohen's d,<< 1 x 10^15,Robust Cohen's d,9.10322,null_CCG uni-class pred- Train IA test RS - Ea...,d,d=9.103
0,3B,Early_IA_Correct,WT VEH,Het VEH,SVM accuracy,10000,10000,0.4457,0.0072,0.4566,0.0240,Robust Cohen's d,0.2701,Robust Cohen's d,0.61239,"null_CCGP- Train on IA trials, Test on RS tria...",d,d=0.612
1,3B,Early_IA_Correct,Het VEH,Het CLNZ,SVM accuracy,10000,10000,0.4566,0.0240,0.5675,0.0179,Robust Cohen's d,8.13 x 10^-08,Robust Cohen's d,5.23761,"null_CCGP- Train on IA trials, Test on RS tria...",d,d=5.238
2,3B,Early_IA_Correct,Het VEH,Het postCLNZ,SVM accuracy,10000,10000,0.4566,0.0240,0.5456,0.0142,Robust Cohen's d,3.18 x 10^-06,Robust Cohen's d,4.51423,"null_CCGP- Train on IA trials, Test on RS tria...",d,d=4.514
3,3B,Early_IA_Error,WT VEH,Het VEH,SVM accuracy,10000,10000,0.4873,0.0137,0.4175,0.0099,Robust Cohen's d,2.66 x 10^-09,Robust Cohen's d,5.83690,"null_CCGP- Train on IA trials, Test on RS tria...",d,d=5.837


In [31]:
full_fig_table['Test p-value']

KeyError: 'Test p-value'